# Reviewer checks: are the synergies non-additive, and are synthetic features collinear with their constituents?

Reviewer 2, comment 2 (and Reviewer 1, comment 2): does MDFS-2D identify genuinely non-additive interactions, or pairs of two individually informative, correlated features? Reviewer 2, comment 5: collinearity between synthetic features and their constituents in the combined model. Reviewer 1, comment 5: cohort-adjusted models.

Three analyses:
1. **Interaction information** of every pair kept by the LOCO pipeline: `II = total_IG − base_IG_1D − contributing_IG_1D` = I(X1,X2;Y) − I(X1;Y) − I(X2;Y). II > 0 means the pair carries more information about the label than the two features add up to (synergy in the information-theoretic sense); II < 0 means redundancy. Because the pipeline's synergy filter only requires `total_IG > max(1D IGs)`, this is a stricter criterion that was not used for selection.
2. **Cohort-adjusted logistic regression with likelihood-ratio tests** for the stable pairs (Results 2): on the pooled 15 two-class CRC cohorts, `y ~ cohort + log x1 + log x2` versus `y ~ cohort + log x1 + log x2 + log x1 × log x2` (interaction LRT), and `y ~ cohort` versus `y ~ cohort + log x1 + log x2` (does the pair carry signal beyond cohort membership?).
3. **Collinearity**: Spearman correlation between each stable synthetic feature (LR or GM, the encoding used in Figure 3) and its two constituents, pooled over the same cohorts.

Inputs: `../M2_loco_crc_pipeline/results/<dt>/mdfs_ig_info_<dt>.csv`, `../R2_classification_performance/final_synthetic_features_table.csv`, `../R2_classification_performance/meta_results_all_pairs.csv`, prepared cohort matrices. Outputs: `interaction_information.tsv`, `stable_pairs_lrt.tsv`, `collinearity.tsv`, `r5_numbers.tsv`.


In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import spearmanr, chi2
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')
RES = '../M2_loco_crc_pipeline/results'
PREP = '../M1_data_preparation/prepared_data'
numbers = []
def note(claim, value, source): numbers.append({'section': 'R5', 'claim': claim, 'value': value, 'source': source}); print(f'{claim}: {value}')


In [ ]:
# 1. Interaction information of the pairs kept by the LOCO pipeline (per-run top-2 % pairs, every fold x run)
rows = []
for dt in ('taxa', 'functions', 'combined'):
    d = pd.read_csv(f'{RES}/{dt}/mdfs_ig_info_{dt}.csv')
    d['II'] = d['total_IG'] - d['base_IG_1D'] - d['contributing_IG_1D']
    d['key'] = [tuple(sorted((a, b))) for a, b in zip(d.base, d.contributing)]
    # one value per unordered pair, fold and run (II is symmetric)
    u = d.drop_duplicates(['fold', 'run', 'key'])
    # consensus pairs of each fold: in the top 2 % of >= 6 of the 10 runs
    cnt = u.groupby(['fold', 'key']).run.nunique()
    cons = cnt[cnt >= 6].reset_index()[['fold', 'key']]
    uc = u.merge(cons, on=['fold', 'key'])
    rows.append({'data_type': dt, 'pair_rows': len(u), 'share_II_positive_all_top2pct': (u.II > 0).mean(),
                 'median_II_all_top2pct': u.II.median(), 'consensus_pair_rows': len(uc),
                 'share_II_positive_consensus': (uc.II > 0).mean(), 'median_II_consensus': uc.II.median(),
                 'median_II_over_total_IG_consensus': (uc.II / uc.total_IG).median()})
    note(f'{dt}: consensus pairs with positive interaction information (joint IG > sum of 1D IGs)',
         f'{100 * (uc.II > 0).mean():.1f} % of {len(uc):,} pair-fold entries (median II {uc.II.median():.1f} bits, {100 * (uc.II / uc.total_IG).median():.0f} % of the pair total IG)',
         f'M2 results/{dt}/mdfs_ig_info_{dt}.csv')
ii = pd.DataFrame(rows); ii.to_csv('interaction_information.tsv', sep='\t', index=False); ii.round(3)


In [ ]:
# 2. Cohort-adjusted logistic regression + LRT for the stable pairs
fin = pd.read_csv('../R2_classification_performance/final_synthetic_features_table.csv')
meta = pd.read_csv('../R2_classification_performance/meta_results_all_pairs.csv')[['data_type', 'feature_pair', 'feature_type']]
fin = fin.merge(meta.rename(columns={'data_type': 'Data Type', 'feature_pair': 'Feature'}), on=['Data Type', 'Feature'], how='left')
ALL = sorted({os.path.basename(p)[2:].rsplit('_', 1)[0] for p in glob.glob(f'{PREP}/X_*_taxa.tsv')})
_cache = {}
def load(dt):
    if dt in _cache: return _cache[dt]
    Xs, ys, cs = [], [], []
    for c in ALL:
        xf, yf = f'{PREP}/X_{c}_{dt}.tsv', f'{PREP}/y_{c}_{dt}.tsv'
        if not (os.path.exists(xf) and os.path.exists(yf)): continue
        X = pd.read_csv(xf, sep='\t', index_col=0); y = pd.read_csv(yf, sep='\t', index_col=0).squeeze().astype(int)
        if y.nunique() < 2: continue                      # single-class cohorts cannot inform a case-control model
        X = X.div(X.sum(axis=1), axis=0); common = X.index.intersection(y.index)
        Xs.append(X.loc[common]); ys.append(y.loc[common]); cs.append(pd.Series(c, index=common))
    X = pd.concat(Xs).fillna(0); _cache[dt] = (X, pd.concat(ys), pd.concat(cs)); return _cache[dt]
def lrt(ll0, ll1, df): return chi2.sf(2 * (ll1 - ll0), df)
def fit(y, Xd):
    try: return sm.Logit(y, Xd).fit(disp=0, maxiter=200, method='bfgs').llf
    except Exception: return np.nan
rows = []
for _, r in fin.iterrows():
    X, y, c = load(r['Data Type']); f1, f2 = r['Feature1'], r['Feature2']
    if f1 not in X.columns or f2 not in X.columns: rows.append({'Data Type': r['Data Type'], 'Feature': r['Feature'], 'note': 'feature missing'}); continue
    x1, x2 = np.log10(X[f1] + 1e-6), np.log10(X[f2] + 1e-6)
    case = (y == 0).astype(int)                            # 0 = CRC in the prepared labels
    C = pd.get_dummies(c, drop_first=True).astype(float)
    base = sm.add_constant(C)
    add = base.assign(x1=x1.values, x2=x2.values)
    inter = add.assign(x1x2=(x1 * x2).values)
    l0, l1, l2 = fit(case, base), fit(case, add), fit(case, inter)
    rows.append({'Data Type': r['Data Type'], 'Feature': r['Feature'], 'Feature1': f1, 'Feature2': f2, 'n_samples': len(y), 'n_cohorts': c.nunique(),
                 'p_pair_beyond_cohort': lrt(l0, l1, 2), 'p_interaction': lrt(l1, l2, 1), 'llf_cohort': l0, 'llf_additive': l1, 'llf_interaction': l2})
lr = pd.DataFrame(rows)
ok = lr.p_interaction.notna()
lr.loc[ok, 'q_interaction'] = multipletests(lr.loc[ok, 'p_interaction'], method='fdr_bh')[1]
lr.loc[ok, 'q_pair_beyond_cohort'] = multipletests(lr.loc[ok, 'p_pair_beyond_cohort'], method='fdr_bh')[1]
lr.to_csv('stable_pairs_lrt.tsv', sep='\t', index=False)
note('Stable pairs informative beyond cohort membership (LRT additive vs cohort-only, BH q < 0.05)', f'{(lr.q_pair_beyond_cohort < 0.05).sum()}/{ok.sum()}', 'R5 stable_pairs_lrt.tsv')
note('Stable pairs with a significant multiplicative interaction term (LRT interaction vs additive, BH q < 0.05)', f'{(lr.q_interaction < 0.05).sum()}/{ok.sum()}', 'R5 stable_pairs_lrt.tsv')
lr.sort_values('p_interaction').head(15)


In [ ]:
# 3. Collinearity of the stable synthetic features with their constituents
rows = []
for _, r in fin.iterrows():
    X, y, c = load(r['Data Type']); f1, f2 = r['Feature1'], r['Feature2']
    if f1 not in X.columns or f2 not in X.columns: continue
    a, b = X[f1].values + 1e-9, X[f2].values + 1e-9
    ft = r['feature_type'] if isinstance(r.get('feature_type'), str) else 'GM'
    s = np.log(a / b) if ft == 'LR' else np.sqrt(a * b)
    rows.append({'Data Type': r['Data Type'], 'Feature': r['Feature'], 'transformation': ft,
                 'spearman_synthetic_vs_f1': spearmanr(s, a).correlation, 'spearman_synthetic_vs_f2': spearmanr(s, b).correlation,
                 'spearman_f1_vs_f2': spearmanr(a, b).correlation})
col = pd.DataFrame(rows); col['max_abs_corr_with_constituent'] = col[['spearman_synthetic_vs_f1', 'spearman_synthetic_vs_f2']].abs().max(axis=1)
col.to_csv('collinearity.tsv', sep='\t', index=False)
note('Stable synthetic features: median |Spearman| with the more correlated constituent (and share > 0.9)',
     f'{col.max_abs_corr_with_constituent.median():.2f} ({100 * (col.max_abs_corr_with_constituent > 0.9).mean():.0f} %)', 'R5 collinearity.tsv')
note('Stable pairs: median Spearman correlation between the two constituents', f'{col.spearman_f1_vs_f2.median():.2f}', 'R5 collinearity.tsv')
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(col.max_abs_corr_with_constituent, bins=20, color='steelblue'); ax[0].set_xlabel('max |Spearman| synthetic vs constituent'); ax[0].set_ylabel('stable pairs')
ax[1].hist(col.spearman_f1_vs_f2, bins=20, color='coral'); ax[1].set_xlabel('Spearman constituent 1 vs 2')
plt.tight_layout(); plt.savefig('collinearity.png', dpi=150); plt.show()
pd.DataFrame(numbers).to_csv('r5_numbers.tsv', sep='\t', index=False, mode='w')
